# Lá sầu riêng: fine-tune trên Colab, validation-only

Chọn **Runtime → Change runtime type → T4 GPU** trong runtime miễn phí, rồi **Runtime → Run all**. Notebook không mua runtime, không mount Drive và không yêu cầu Kaggle API key.

Protocol cố định: pretrained MobileNetV3Small; seed1337; batch64; tối đa24epochs / 30phút training; early stopping và checkpoint theo validation macro-F1 như pipeline đã audit. Đây là run mới theo protocol dài hơn, không phải test độc lập: **test set đã được xem trong run local trước nên lần này không evaluate test, không fit calibration/threshold và không báo selective accuracy**. Validation được dùng chọn checkpoint nên không dùng làm accuracy thực tế ngoài vườn. Không thay model production.

Dataset [DurianLDD v1, cthng123](https://www.kaggle.com/datasets/cthng123/durian-leaf-disease-dataset), CC BY4.0; attribution ĐứcThắng123 / Thien B.Nguyen-Tat / Duc Thang Nong. Giữ source split sau exact/near-duplicate grouping; không có orchard/tree ID. Script mới được embed nguyên bản và khóa SHA256; source gốc tham chiếu commit `bfce1662da352cb6394c4e7bc3284664d4b8c5d4`. GPU/Linux có thể khác số so với RTX4060/Windows.


In [ ]:
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import urllib.request
from zipfile import ZipFile, ZIP_DEFLATED
import stat

WORK = Path('/content/durian_leaf_colab')
WORK.mkdir(parents=True, exist_ok=True)
if not shutil.which('nvidia-smi'):
    raise RuntimeError('GPU chưa được cấp. Chọn T4 GPU miễn phí trước khi Run all; không chạy CPU.')
gpu_info = subprocess.check_output(['nvidia-smi'], text=True)
print(gpu_info)
max_cuda = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', gpu_info)
if max_cuda and tuple(map(int, max_cuda.groups())) < (12, 8):
    raise RuntimeError('Driver runtime này chưa hỗ trợ CUDA12.8 đã khóa; dừng để kiểm runtime miễn phí tương thích.')
if sys.version_info < (3, 12):
    raise RuntimeError('Dependencies đã audit cần Python3.12 trở lên. Kiểm phiên bản runtime trước khi tiếp tục.')

def stream_checked(command, cwd=None, log=None):
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            if log is not None:
                log.write(line)
                log.flush()
        exit_code = process.wait()
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        process.stdout.close()
    if exit_code:
        raise subprocess.CalledProcessError(exit_code, command)

# Venv riêng tránh sửa packages của notebook kernel; reuse đúng versions nếu có.
VENV = WORK/'venv'
PYTHON = VENV/'bin/python'
stream_checked([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(VENV)])
stream_checked([str(PYTHON), '-m', 'pip', '--version'])
version_probe = "import importlib.metadata as m,json; names=('torch','torchvision','numpy','pillow','onnx','onnxruntime'); print(json.dumps({name:next((d.version for d in m.distributions(name=name)),None) for name in names}))"
versions = json.loads(subprocess.check_output([str(PYTHON), '-c', version_probe], text=True))
TORCH_CHECK = "import torch,torchvision; assert torch.__version__ == '2.8.0+cu128', torch.__version__; assert torchvision.__version__ == '0.23.0+cu128', torchvision.__version__; assert torch.cuda.is_available(), 'PyTorch không thấy CUDA; dừng CPU fallback'; print('GPU:',torch.cuda.get_device_name(0),'torch:',torch.__version__,'vision:',torchvision.__version__)"
torch_ready = versions.get('torch') == '2.8.0+cu128' and versions.get('torchvision') == '0.23.0+cu128'
if torch_ready:
    probe = subprocess.run([str(PYTHON), '-c', TORCH_CHECK], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True)
    print(probe.stdout, flush=True)
    torch_ready = probe.returncode == 0
if not torch_ready:
    # Copy every Torch/CUDA dependency into this venv; global namespace packages are insufficient.
    stream_checked([str(PYTHON), '-m', 'pip', 'install', '--ignore-installed',
                    'torch==2.8.0+cu128', 'torchvision==0.23.0+cu128',
                    '--index-url', 'https://download.pytorch.org/whl/cu128', '--disable-pip-version-check'])
pins = {'numpy':'2.5.2', 'pillow':'12.3.0', 'onnx':'1.19.0', 'onnxruntime':'1.23.0'}
versions = json.loads(subprocess.check_output([str(PYTHON), '-c', version_probe], text=True))
missing = [name+'=='+version for name, version in pins.items() if versions.get(name) != version]
if missing:
    stream_checked([str(PYTHON), '-m', 'pip', 'install', *missing, '--disable-pip-version-check'])
stream_checked([str(PYTHON), '-c', TORCH_CHECK])
os.environ['TORCH_HOME'] = str(WORK/'torch-cache')
print('GPU và dependencies sẵn sàng.')


## Source snapshot và dataset

Chỉ tải public GitHub/Kaggle. Nếu response bị đổi, auth/CAPTCHA hoặc SHA256 khác, cell dừng; không bypass. Notebook TF cũ chỉ là fixture cho preprocessing self-check, **không chạy training notebook đó**.


In [ ]:
COMMIT = 'bfce1662da352cb6394c4e7bc3284664d4b8c5d4'
RAW = 'https://raw.githubusercontent.com/vinhelysia/DurainCheckerDemo/'+COMMIT+'/'
SCRIPT_SHA256 = '3c301d8bc959b699a4cad2f94fc6799abe6f9f70a004c17073ce7bfd77c7e6fc'
PIPELINE_SOURCE = '"""Reproducible leaf candidate: prepare -> train. Never writes production models."""\nimport argparse\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport time\nfrom zipfile import ZipFile\n\nimport numpy as np\nfrom PIL import Image, ImageOps\n\nCLASSES = ["HEALTHY_LEAF", "ALGAL_LEAF_SPOT", "LEAF_BLIGHT",\n           "PHOMOPSIS_LEAF_SPOT", "ALLOCARIDARA_ATTACK"]\nSPLITS = ["train", "val", "test"]\nSOURCE = "https://www.kaggle.com/datasets/cthng123/durian-leaf-disease-dataset"\nARCHIVE_SHA256 = "d90b6f9d856d3f48016d78a205bc028d113e848fce1e07a072d1fe03ae0ce3bd"\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef save(path, data):\n    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n\n\ndef image_hashes(path):\n    with Image.open(path) as original:\n        image = ImageOps.exif_transpose(original).convert("RGB")\n        pixel_hash = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()\n        small = np.asarray(image.convert("L").resize((9, 8), Image.Resampling.LANCZOS))\n        dhash = sum(int(bit) << i for i, bit in enumerate((small[:, 1:] > small[:, :-1]).ravel()))\n    return pixel_hash, dhash\n\n\ndef deduplicate(records, radius=4):\n    """Preserve source test over validation over train; groups never cross splits."""\n    parent = list(range(len(records)))\n\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n\n    # ponytail: O(n²) for this 4,437-image dataset; use a BK-tree for much larger corpora.\n    for i, a in enumerate(records):\n        for j in range(i):\n            b = records[j]\n            if a["pixel_sha256"] == b["pixel_sha256"] or (a["dhash"] ^ b["dhash"]).bit_count() <= radius:\n                parent[find(i)] = find(j)\n    groups = defaultdict(list)\n    for i, record in enumerate(records):\n        groups[find(i)].append(record)\n    kept, excluded = [], []\n    for members in groups.values():\n        split = max((r["split"] for r in members), key=SPLITS.index)\n        exact = {}\n        for record in sorted(members, key=lambda r: (-SPLITS.index(r["split"]), r["path"])):\n            previous = exact.get(record["pixel_sha256"])\n            if previous and previous["label"] != record["label"]:\n                raise ValueError(f"Identical pixels have conflicting labels: {previous[\'path\']} / {record[\'path\']}")\n            exact[record["pixel_sha256"]] = previous or record\n            if previous or record["split"] != split:\n                excluded.append({**record, "reason": "exact_duplicate" if previous else "near_duplicate_cross_split"})\n            else:\n                kept.append({**record, "group": members[0]["path"]})\n    return kept, excluded\n\n\ndef check_manifest(manifest):\n    if manifest["classes"] != CLASSES or not manifest["records"]:\n        raise ValueError("Empty dataset or wrong class order")\n    groups, pixels, paths = {}, set(), set()\n    counts = Counter()\n    for record in manifest["records"]:\n        if record["split"] not in SPLITS or record["label"] not in CLASSES:\n            raise ValueError("Unknown split/class")\n        if record["pixel_sha256"] in pixels or record["path"] in paths:\n            raise ValueError("Duplicate image in manifest")\n        pixels.add(record["pixel_sha256"])\n        paths.add(record["path"])\n        if groups.setdefault(record["group"], record["split"]) != record["split"]:\n            raise ValueError("Near-duplicate leakage between splits")\n        counts[record["split"], record["label"]] += 1\n    if any(counts[s, c] == 0 for s in SPLITS for c in CLASSES):\n        raise ValueError("Each split must contain every class")\n    return {s: {c: counts[s, c] for c in CLASSES} for s in SPLITS}\n\n\ndef prepare(args):\n    root, output = args.dataset.resolve(), args.output.resolve()\n    if sha(args.archive) != ARCHIVE_SHA256:\n        raise ValueError("Unexpected source archive: re-audit license/version before changing the locked SHA256")\n    output.mkdir(parents=True, exist_ok=True)\n    target = output / "manifest.json"\n    if target.exists():\n        raise ValueError("Manifest is locked: use a new output directory to prepare another dataset")\n    records = []\n    for path in sorted(root.rglob("*")):\n        if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:\n            continue\n        label, source_split = path.parent.name, path.parent.parent.name.lower()\n        split = "val" if source_split in {"val", "valid", "validation"} else source_split\n        if label not in CLASSES or split not in SPLITS:\n            raise ValueError(f"Unexpected class/split folder: {path}")\n        pixel, dhash = image_hashes(path)\n        records.append({"path": path.relative_to(root).as_posix(), "label": label, "split": split,\n                        "sha256": sha(path), "pixel_sha256": pixel, "dhash": dhash})\n    with ZipFile(args.archive) as archive:\n        if set(archive.namelist()) != {r["path"] for r in records}:\n            raise ValueError("Dataset file list differs from the source archive")\n        for record in records:\n            if hashlib.sha256(archive.read(record["path"])).hexdigest() != record["sha256"]:\n                raise ValueError(f"Dataset image differs from source archive: {record[\'path\']}")\n    kept, excluded = deduplicate(records)\n    manifest = {"source": SOURCE, "version": 1, "license": "CC BY 4.0", "classes": CLASSES,\n                "archive_sha256": sha(args.archive), "source_images": len(records),\n                "deduplication": "exact RGB pixels + 64-bit dHash Hamming <=4; holdout-priority groups",\n                "split_policy": "Source splits preserved; discard cross-split group members from lower-priority splits",\n                "orchard_tree_leaf_ids": "not provided; image holdout only, no independent-orchard claim",\n                "records": kept, "excluded": excluded}\n    manifest["counts"] = check_manifest(manifest)\n    save(target, manifest)\n    (output / "manifest.sha256").write_text(sha(target) + "\\n", encoding="ascii")\n    print(json.dumps({"images": len(records), "kept": len(kept), "excluded": len(excluded),\n                      "counts": manifest["counts"], "manifest_sha256": sha(target)}, indent=2), flush=True)\n\n\ndef metrics(probabilities, labels, threshold=0.0, margin=0.0):\n    if (probabilities.shape != (len(labels), len(CLASSES)) or not len(labels)\n            or not np.isfinite(probabilities).all() or (probabilities < 0).any()\n            or not np.allclose(probabilities.sum(1), 1, atol=1e-5)):\n        raise ValueError("Metrics require finite five-class probability distributions and non-empty labels")\n    predictions = probabilities.argmax(1)\n    cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)\n    for truth, prediction in zip(labels, predictions):\n        cm[truth, prediction] += 1\n    recall = np.diag(cm) / np.maximum(cm.sum(1), 1)\n    precision = np.diag(cm) / np.maximum(cm.sum(0), 1)\n    f1 = 2 * recall * precision / np.maximum(recall + precision, 1e-12)\n    confidence = probabilities.max(1)\n    correct = predictions == labels\n    ece = 0.0\n    for low in np.linspace(0, .9, 10):\n        mask = (confidence >= low) & ((confidence < low + .1) if low < .9 else (confidence <= 1))\n        if mask.any():\n            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())\n    sorted_probs = np.sort(probabilities, axis=1)\n    accepted = (confidence >= threshold) & ((sorted_probs[:, -1] - sorted_probs[:, -2]) >= margin)\n    return {"n": len(labels), "accuracy": float(correct.mean()), "macro_f1": float(f1.mean()),\n            "per_class_recall": dict(zip(CLASSES, recall.tolist())), "confusion_matrix": cm.tolist(),\n            "nll": float(-np.log(np.maximum(probabilities[np.arange(len(labels)), labels], 1e-12)).mean()),\n            "brier": float(np.square(probabilities - np.eye(len(CLASSES))[labels]).sum(1).mean()),\n            "ece_10_bins": float(ece), "abstention": {"threshold": threshold, "margin": margin,\n            "accepted": int(accepted.sum()), "coverage": float(accepted.mean()),\n            "accepted_accuracy": float(correct[accepted].mean()) if accepted.any() else None}}\n\n\ndef wilson_lower(correct, total):\n    if not total:\n        return 0.0\n    z = 1.96\n    p = correct / total\n    return (p + z*z/(2*total) - z * math.sqrt(p*(1-p)/total + z*z/(4*total*total))) / (1 + z*z/total)\n\n\ndef train(args):\n    import os\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    import torch\n    from torchvision import models, transforms\n    from torch.utils.data import Dataset, DataLoader\n    import onnxruntime as ort\n\n    out, root = args.output.resolve(), args.dataset.resolve()\n    manifest_path = out / "manifest.json"\n    if sha(manifest_path) != (out / "manifest.sha256").read_text().strip():\n        raise ValueError("Manifest changed after locking")\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    check_manifest(manifest)\n    if sha(args.archive) != manifest["archive_sha256"]:\n        raise ValueError("Archive changed after locking")\n    for record in manifest["records"]:\n        path = (root / record["path"]).resolve()\n        if not path.is_relative_to(root) or sha(path) != record["sha256"]:\n            raise ValueError(f"Image changed or escaped dataset root: {record[\'path\']}")\n    if args.epochs < 2 or args.batch_size < 1 or args.minutes <= 0:\n        raise ValueError("Require epochs >=2, positive batch size and time budget")\n    if (out / "metrics.json").exists():\n        raise ValueError("This run already has metrics. Preserve it; use a new run directory for a deliberate new protocol.")\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    torch.set_num_threads(4)\n    torch.use_deterministic_algorithms(True, warn_only=True)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if device.type != "cuda":\n        raise RuntimeError("This local training run requires CUDA; no silent slow CPU fallback")\n    torch.backends.cudnn.benchmark = False\n    print(f"GPU: {torch.cuda.get_device_name(0)}; torch {torch.__version__}", flush=True)\n\n    class LeafDataset(Dataset):\n        def __init__(self, split):\n            self.records = [r for r in manifest["records"] if r["split"] == split]\n            self.images = []\n            for record in self.records:\n                with Image.open(root / record["path"]) as image:\n                    self.images.append(ImageOps.exif_transpose(image).convert("RGB").resize((224, 224), Image.Resampling.BILINEAR))\n            augmentation = [transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),\n                            transforms.ColorJitter(brightness=.1, contrast=.1, saturation=.1)] if split == "train" else []\n            self.transform = transforms.Compose(augmentation + [transforms.ToTensor()])\n        def __len__(self):\n            return len(self.records)\n        def __getitem__(self, index):\n            return self.transform(self.images[index]).permute(1, 2, 0), CLASSES.index(self.records[index]["label"])\n\n    active_splits = ["train", "val"] if args.validation_only else SPLITS\n    datasets = {s: LeafDataset(s) for s in active_splits}\n    loaders = {s: DataLoader(datasets[s], batch_size=args.batch_size, shuffle=s == "train", num_workers=0) for s in active_splits}\n    backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)\n    backbone.classifier[-1] = torch.nn.Linear(backbone.classifier[-1].in_features, len(CLASSES))\n\n    class Candidate(torch.nn.Module):\n        def __init__(self, model):\n            super().__init__()\n            self.model = model\n            self.register_buffer("mean", torch.tensor([.485, .456, .406]).reshape(1, 3, 1, 1))\n            self.register_buffer("std", torch.tensor([.229, .224, .225]).reshape(1, 3, 1, 1))\n        def forward(self, rgb):\n            return self.model((rgb.permute(0, 3, 1, 2) - self.mean) / self.std)\n\n    model = Candidate(backbone).to(device)\n    for parameter in backbone.features.parameters():\n        parameter.requires_grad = False\n    counts = Counter(r["label"] for r in datasets["train"].records)\n    weights = torch.tensor([len(datasets["train"])/(len(CLASSES)*counts[c]) for c in CLASSES], device=device)\n    criterion = torch.nn.CrossEntropyLoss(weight=weights)\n    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)\n\n    def predict(loader):\n        model.eval()\n        logits, labels = [], []\n        with torch.inference_mode():\n            for images, targets in loader:\n                logits.append(model(images.to(device)).cpu())\n                labels.append(targets)\n        return torch.cat(logits), torch.cat(labels).numpy()\n\n    best, stale, history = -1.0, 0, []\n    started = time.monotonic()\n    checkpoint = out / "candidate.pt"\n    for epoch in range(args.epochs):\n        if epoch == 2:\n            for parameter in backbone.features[-4:].parameters():\n                parameter.requires_grad = True\n            optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-5, weight_decay=1e-4)\n        model.train()\n        # Keep pretrained batch statistics fixed for this small dataset.\n        for layer in model.modules():\n            if isinstance(layer, torch.nn.BatchNorm2d):\n                layer.eval()\n        losses = []\n        for images, targets in loaders["train"]:\n            optimizer.zero_grad(set_to_none=True)\n            loss = criterion(model(images.to(device)), targets.to(device))\n            loss.backward()\n            optimizer.step()\n            losses.append(float(loss.detach()))\n        logits, labels = predict(loaders["val"])\n        report = metrics(logits.softmax(1).numpy(), labels)\n        item = {"epoch": epoch + 1, "train_loss": float(np.mean(losses)), "val_accuracy": report["accuracy"],\n                "val_macro_f1": report["macro_f1"], "elapsed_seconds": time.monotonic() - started}\n        history.append(item)\n        save(out / "history.json", history)\n        print(json.dumps(item), flush=True)\n        if report["macro_f1"] > best:\n            best, stale = report["macro_f1"], 0\n            torch.save({"state_dict": model.state_dict(), "epoch": epoch + 1}, checkpoint)\n        else:\n            stale += 1\n        if epoch >= 3 and (stale >= 4 or time.monotonic() - started >= args.minutes * 60):\n            break\n    selected = torch.load(checkpoint, map_location=device, weights_only=True)\n    model.load_state_dict(selected["state_dict"])\n    val_logits, val_labels = predict(loaders["val"])\n    temperature, threshold, margin = 1.0, None, None\n    if not args.validation_only:\n        temperature = min(np.linspace(.5, 3.0, 51), key=lambda t: metrics((val_logits/t).softmax(1).numpy(), val_labels)["nll"])\n        val_probs = (val_logits/temperature).softmax(1).numpy()\n        threshold, margin = 1.01, .2\n        for value in np.linspace(.5, .99, 50):\n            ordered = np.sort(val_probs, axis=1)\n            accepted = (ordered[:, -1] >= value) & (ordered[:, -1] - ordered[:, -2] >= margin)\n            successes = int((val_probs.argmax(1)[accepted] == val_labels[accepted]).sum())\n            if accepted.sum() >= 30 and wilson_lower(successes, int(accepted.sum())) >= .9:\n                threshold = float(value)\n                break\n    config = {"seed": args.seed, "epochs_limit": args.epochs, "batch_size": args.batch_size, "minutes": args.minutes,\n              "selected_epoch": selected["epoch"], "temperature": float(temperature), "confidence_threshold": threshold,\n              "margin_threshold": margin, "validation_only": args.validation_only,\n              "selection": "validation macro-F1 only" if args.validation_only else "validation macro-F1; temperature/threshold validation only",\n              "manifest_sha256": sha(manifest_path), "input": "float32 NHWC RGB [0,1] 1x224x224x3",\n              "backbone": "torchvision MobileNetV3Small ImageNet1K_V1", "torch": torch.__version__}\n    save(out / "training-config.json", config)\n    if args.validation_only:\n        validation = metrics(val_logits.softmax(1).numpy(), val_labels)\n        validation.pop("abstention")\n        result = {"validation_raw": validation, "config": config,\n                  "evaluation_status": "validation_only; reused test holdout not evaluated",\n                  "calibration_status": "not fitted; no selective accuracy or abstention policy evaluated"}\n        np.savez_compressed(out / "predictions.npz", val_logits=val_logits.numpy(), val_labels=val_labels)\n    else:\n        # Configuration is frozen before the first and only candidate test evaluation.\n        test_logits, test_labels = predict(loaders["test"])\n        test_probs = (test_logits/temperature).softmax(1).numpy()\n        result = {"validation_raw": metrics(val_logits.softmax(1).numpy(), val_labels),\n                  "validation_calibrated": metrics(val_probs, val_labels, threshold, margin),\n                  "test_raw": metrics(test_logits.softmax(1).numpy(), test_labels),\n                  "test_calibrated": metrics(test_probs, test_labels, threshold, margin), "config": config}\n        np.savez_compressed(out / "predictions.npz", val_logits=val_logits.numpy(), val_labels=val_labels,\n                            test_logits=test_logits.numpy(), test_labels=test_labels)\n        # Old model provenance is unknown; this comparison cannot prove absence of its test leakage.\n        session = ort.InferenceSession(str(args.baseline), providers=["CPUExecutionProvider"])\n        for split, labels in [("val", val_labels), ("test", test_labels)]:\n            baseline_probs = []\n            for index in range(len(datasets[split])):\n                pixels = np.asarray(datasets[split].images[index], dtype=np.float32)[None] / 255.0\n                baseline_probs.append(session.run(None, {session.get_inputs()[0].name: pixels})[0][0])\n            result["baseline_" + ("validation" if split == "val" else "test")] = metrics(np.asarray(baseline_probs), labels, .8, .2)\n        result["baseline_model_sha256"] = sha(args.baseline)\n        result["baseline_provenance_warning"] = "Existing model training data unknown; possible exposure to this source holdout"\n    save(out / "metrics.json", result)\n\n    class Export(torch.nn.Module):\n        def __init__(self, trained):\n            super().__init__()\n            self.trained = trained\n        def forward(self, image):\n            return torch.softmax(self.trained(image) / float(temperature), dim=1)\n\n    exported = Export(model.cpu()).eval()\n    onnx_path = out / "candidate.onnx"\n    dummy = torch.zeros(1, 224, 224, 3)\n    torch.onnx.export(exported, dummy, str(onnx_path), input_names=["input"], output_names=["probabilities"],\n                      opset_version=17, dynamo=False)\n    candidate_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])\n    exported.eval()\n    difference = 0.0\n    for image in datasets["val"].images[:5]:\n        sample = np.asarray(image, dtype=np.float32)[None] / 255.0\n        with torch.inference_mode():\n            expected = exported(torch.from_numpy(sample)).numpy()\n        actual = candidate_session.run(None, {"input": sample})[0]\n        difference = max(difference, float(np.abs(expected - actual).max()))\n        if difference > 1e-4 or actual.shape != (1, len(CLASSES)) or not np.isfinite(actual).all():\n            raise ValueError(f"ONNX parity failed: {difference}")\n    save(out / "export.json", {"onnx_sha256": sha(onnx_path), "max_absolute_difference": difference,\n                               "validation_samples_checked": 5,\n                               "classes": CLASSES, "status": "candidate_only_not_promoted"})\n    print(json.dumps(result, indent=2), flush=True)\n\n\ndef self_check():\n    records = [{"path": f"{s}/{c}/a.jpg", "label": c, "split": s, "pixel_sha256": f"{s}-{c}",\n                "dhash": (i + 1) * 0x123456789ABCDEF} for i, (s, c) in enumerate((s,c) for s in SPLITS for c in CLASSES)]\n    records.append({**records[0], "path": "test/HEALTHY_LEAF/duplicate.jpg", "split": "test"})\n    kept, excluded = deduplicate(records, radius=0)\n    assert len(excluded) == 1 and excluded[0]["split"] == "train"\n    record = {"path": "extra", "label": CLASSES[0], "split": "train", "pixel_sha256": "extra", "dhash": 1, "group": "extra"}\n    clean = [{**r, "group": r["path"]} for r in records[:-1]] + [record]\n    check_manifest({"classes": CLASSES, "records": clean})\n    bad = [{**r} for r in clean]\n    bad[-1]["group"] = bad[-2]["group"]\n    try:\n        check_manifest({"classes": CLASSES, "records": bad})\n    except ValueError:\n        pass\n    else:\n        raise AssertionError("Leakage check accepted overlapping groups")\n    perfect = metrics(np.eye(5), np.arange(5), .8, .2)\n    assert perfect["macro_f1"] == 1 and perfect["ece_10_bins"] == 0 and perfect["abstention"]["accepted"] == 5\n    assert wilson_lower(100, 100) > .9 and wilson_lower(5, 5) < .9\n    notebook = json.loads(Path(__file__).with_name("durian_leaf_disease_classifier_colab.ipynb").read_text(encoding="utf-8"))\n    source = "\\n".join("".join(c["source"]) for c in notebook["cells"])\n    assert \'Rescaling(2.0, offset=-1.0\' in source and \'x = base_model(x, training=False)\' in source\n    print("Self-check passed: holdout priority, group leakage, class order, metrics, preprocessing")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("command", choices=["prepare", "train", "self-check"])\n    parser.add_argument("--dataset", type=Path, default=Path("tmp/ai-training/dataset"))\n    parser.add_argument("--archive", type=Path, default=Path("tmp/ai-training/durian-leaf-v1.zip"))\n    parser.add_argument("--output", type=Path, default=Path("tmp/ai-training/run-1"))\n    parser.add_argument("--baseline", type=Path, default=Path("api/leaf_disease_model.onnx"))\n    parser.add_argument("--validation-only", action="store_true",\n                        help="Train/select on validation; do not evaluate previously inspected test holdout or fit calibration")\n    parser.add_argument("--seed", type=int, default=1337)\n    parser.add_argument("--epochs", type=int, default=12)\n    parser.add_argument("--batch-size", type=int, default=64)\n    parser.add_argument("--minutes", type=float, default=12)\n    args = parser.parse_args()\n    {"prepare": prepare, "train": train, "self-check": lambda _: self_check()}[args.command](args)\n'

def download_locked(url, path, expected_sha256):
    if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected_sha256:
        return
    temporary = path.with_suffix(path.suffix+'.part')
    request = urllib.request.Request(url, headers={'User-Agent':'DurianLeafColab/1.0'})
    with urllib.request.urlopen(request, timeout=120) as response, temporary.open('wb') as target:
        shutil.copyfileobj(response, target)
    actual = hashlib.sha256(temporary.read_bytes()).hexdigest()
    if actual != expected_sha256:
        raise RuntimeError('Download SHA256 mismatch; kiểm source/version hoặc access wall: '+url)
    temporary.replace(path)

ML = WORK/'ml'
ML.mkdir(exist_ok=True)
download_locked(RAW+'ml/leaf_training.py', ML/'audited_base_leaf_training.py', 'd7c492ae886a4ed6d807d6f7d65f0924bcb29440ebecfa949b625e895db18962')
download_locked(RAW+'ml/durian_leaf_disease_classifier_colab.ipynb', ML/'durian_leaf_disease_classifier_colab.ipynb', '7f20a30efcab209fbf087c5b76420e2c898932a9b7c934728f0fb79ef7272fbb')
SCRIPT = ML/'leaf_training.py'
assert hashlib.sha256(PIPELINE_SOURCE.encode()).hexdigest() == SCRIPT_SHA256
SCRIPT.write_text(PIPELINE_SOURCE, encoding='utf-8')
assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == SCRIPT_SHA256
stream_checked([str(PYTHON), str(SCRIPT), 'self-check'], cwd=WORK)

DATA_DIR = WORK/'tmp/ai-training'
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE = DATA_DIR/'durian-leaf-v1.zip'
DATASET = DATA_DIR/'dataset'
download_locked('https://www.kaggle.com/api/v1/datasets/download/cthng123/durian-leaf-disease-dataset?datasetVersionNumber=1',
                ARCHIVE, 'd90b6f9d856d3f48016d78a205bc028d113e848fce1e07a072d1fe03ae0ce3bd')
DATASET.mkdir(exist_ok=True)
with ZipFile(ARCHIVE) as archive:
    if len(archive.infolist()) != 4437 or sum(item.file_size for item in archive.infolist()) > 100*1024*1024:
        raise RuntimeError('Unexpected archive size/file count')
    for item in archive.infolist():
        target = (DATASET/item.filename).resolve()
        if '\\' in item.filename or not target.is_relative_to(DATASET.resolve()) or stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Unsafe archive path: '+item.filename)
    archive.extractall(DATASET)
OUTPUT = DATA_DIR/'colab-validation-24'
COMMON = ['--dataset', str(DATASET), '--archive', str(ARCHIVE), '--output', str(OUTPUT)]
if not (OUTPUT/'manifest.json').exists():
    stream_checked([str(PYTHON), str(SCRIPT), 'prepare', *COMMON], cwd=WORK)
else:
    print('Giữ manifest đã khóa; train sẽ kiểm SHA256 trước khi dùng.')
print('Dataset public v1 hash verified; source splits/dedup đã khóa.')


## Fine-tune GPU

Cell này chạy tối đa24epochs, dừng theo patience4 hoặc time budget30phút như script. Giữ nguyên hyperparameters; không dùng test đã xem để tuning. Có thể mở cell cuối để tải artifacts sau lỗi export/runtime, nhưng không coi partial run là hoàn tất.


In [ ]:
PROTOCOL = {
    'source_commit': COMMIT, 'pipeline_sha256': SCRIPT_SHA256,
    'epochs_limit': 24, 'seed': 1337, 'batch_size': 64, 'minutes': 30,
    'validation_only': True, 'calibration_fitted': False,
    'evaluation_note': 'Previously inspected test holdout is not evaluated; validation selects checkpoints.',
    'deployment': 'research candidate only; no production replacement',
}
OUTPUT.mkdir(exist_ok=True)
(OUTPUT/'colab-protocol.json').write_text(json.dumps(PROTOCOL, indent=2), encoding='utf-8')
(OUTPUT/'leaf_training.py').write_text(PIPELINE_SOURCE, encoding='utf-8')
(OUTPUT/'gpu.txt').write_text(gpu_info, encoding='utf-8')
(OUTPUT/'environment.txt').write_text(subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
command = [str(PYTHON), '-u', str(SCRIPT), 'train', *COMMON, '--validation-only',
           '--epochs', '24', '--seed', '1337', '--batch-size', '64', '--minutes', '30']
with (OUTPUT/'train.log').open('w', encoding='utf-8') as log:
    stream_checked(command, cwd=WORK, log=log)
print('Training/export hoàn tất. Chỉ có validation metrics; test không evaluate.')


## Kết quả thật và download

Accuracy/F1 dưới đây là **validation dùng chọn checkpoint**, không phải test mới hay accuracy ngoài thực tế. ONNX parity kiểm5validationảnh CPU; không tự promote hoặc gắn nhãn score đã calibrated. Download ZIP để giữ artifacts trước khi Colab ngắt runtime.


In [ ]:
metrics_path, export_path = OUTPUT/'metrics.json', OUTPUT/'export.json'
if metrics_path.exists():
    result = json.loads(metrics_path.read_text())
    assert result['evaluation_status'].startswith('validation_only')
    assert result['config']['validation_only'] is True
    assert result['config']['epochs_limit'] == 24
    assert result['config']['seed'] == 1337 and result['config']['batch_size'] == 64
    assert not any(key.startswith(('test_', 'baseline_')) for key in result)
    assert 'abstention' not in result['validation_raw']
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print('Metrics chưa có; đây là partial artifacts, không có kết quả training để báo.')
if export_path.exists():
    print('ONNX parity:', export_path.read_text())
else:
    print('Export chưa được verify; không dùng candidate.onnx.')
ARTIFACT_ZIP = WORK/'durian-leaf-colab-validation-24.zip'
with ZipFile(ARTIFACT_ZIP, 'w', ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT))
print('Artifact ZIP:', ARTIFACT_ZIP, 'SHA256:', hashlib.sha256(ARTIFACT_ZIP.read_bytes()).hexdigest())
from google.colab import files
files.download(str(ARTIFACT_ZIP))
